# Track A — Bước A4: Thuật toán `Imbalanced-DYDM` + `NIW` Memory cho `GC10-DET`

- Nhánh: `duytung/08-gc10-dydm-imbalanced`
- Sửa trực tiếp 2 nhược điểm của Bước A3: (1) thay Sinkhorn-Knopp chia đều bằng **Prior-Preserving KL + Confidence-Weighted Self-Training** giữ tỷ lệ cụm từ `FTR1`; (2) đổi augmentation sang **giữ toàn khung hình** để không cắt trượt lỗi nhỏ.
- Tái sử dụng `ft_ckpt.pt` + snapshot `s0..s4` từ Bước A0 (`ep=20`) và `/content/GC10-DET` nếu đang dùng chung runtime Colab.

In [ ]:
# 1. Kết nối Google Drive (bỏ qua nếu đã mount sẵn)
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Google Drive đã được mount sẵn!')

In [ ]:
# 2. Cập nhật nhánh duytung/08-gc10-dydm-imbalanced
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'
BRANCH = 'duytung/08-gc10-dydm-imbalanced'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin {BRANCH}
!git checkout -B {BRANCH} origin/{BRANCH}
!git status -sb

In [ ]:
# 3. Cài gói thiếu (bỏ qua nếu runtime hiện tại đã cài ở bước trước)
!pip uninstall -y -q torchao
!pip install -q numpyro continuum peft datasets transformers scipy scikit-learn psutil
!nvidia-smi

In [ ]:
# 4. Chạy nền Bước A4 (Imbalanced-DYDM, 5 seeds s0..s4)
import os, subprocess, time
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a4_imbal_dydm.log'
PID_FILE = f'{WORK_DIR}/gc10_a4_imbal_dydm.pid'

with open(LOG_FILE, 'w') as f:
    p = subprocess.Popen(['bash', 'colab/run_gc10_a4_bg.sh', 'auto'], stdout=f, stderr=subprocess.STDOUT)
with open(PID_FILE, 'w') as f:
    f.write(str(p.pid))
print(f'Đã khởi chạy nền Bước A4 với PID={p.pid}')
print(f'Log: {LOG_FILE}')
time.sleep(3)
!tail -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ & xem kết quả + thông số phần cứng peak
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a4_imbal_dydm.log'
PID_FILE = f'{WORK_DIR}/gc10_a4_imbal_dydm.pid'
SUMMARY_OUT = f'{WORK_DIR}/gc10_a4_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = False
if pid and os.path.exists(f'/proc/{pid}/status'):
    st = open(f'/proc/{pid}/status').read()
    alive = ('State:\tZ' not in st and 'State: Z' not in st)
print(f'Trạng thái PID={pid}: {"ĐANG CHẠY" if alive else "ĐÃ DỪNG / XONG"}')
print('\n--- 40 dòng log gần nhất ---')
if os.path.exists(LOG_FILE):
    lines = open(LOG_FILE).read().splitlines()
    print('\n'.join(lines[-40:]))
if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ & THÔNG SỐ PHẦN CỨNG PEAK BƯỚC A4 (GC10 Imbalanced-DYDM) ---')
    print(open(SUMMARY_OUT).read())